In [28]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer,AutoModel
import torch.nn.functional as F
import torch.optim as optim
from pathlib import Path
from torch.amp import autocast,GradScaler
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}, CUDA devices: {torch.cuda.device_count()}")
print(f"torch version: {torch.__version__}, torch cuda version: {torch.version.cuda}")

device: cuda, CUDA devices: 2
torch version: 2.10.0+cu128, torch cuda version: 12.8


In [2]:
def load_data():
    dataset_path = Path("/kaggle/input/datasets/kaushikrajagiri/english-to-telugu/en-te")
    eng_dataset_path = dataset_path/"train.en"
    tel_dataset_path = dataset_path/"train.te"
    print(dataset_path)
    print(eng_dataset_path)
    print(tel_dataset_path)

    with open(eng_dataset_path,mode="r",encoding="utf-8") as f:
        train_eng = f.read().splitlines()

    with open(tel_dataset_path,mode="r",encoding="utf-8") as f:
        train_tel = f.read().splitlines()

    assert len(train_eng) == len(train_tel)

    return train_eng,train_tel

train_eng,train_tel = load_data()


/kaggle/input/datasets/kaushikrajagiri/english-to-telugu/en-te
/kaggle/input/datasets/kaushikrajagiri/english-to-telugu/en-te/train.en
/kaggle/input/datasets/kaushikrajagiri/english-to-telugu/en-te/train.te


In [3]:
print(len(train_eng),len(train_tel))

4841862 4841862


## tokenize

In [4]:
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

def tokenization_in_batch(texts,tokenizer,Max_length = 128,batch_size = 512):
    all_input_ids, all_attention_masks = [], []
    for i in range(0,len(texts),batch_size):
        batch = texts[i:i+batch_size]
        encoding =  tokenizer(batch,padding = "max_length",
                        truncation=True,
                        max_length=Max_length,
                        return_tensors="pt")
        all_input_ids.append(encoding["input_ids"])
        all_attention_masks.append(encoding["attention_mask"])

    return {"input_ids": torch.cat(all_input_ids, dim=0), "attention_mask": torch.cat(all_attention_masks, dim=0)}
subset_size = 5000
encoded_eng = tokenization_in_batch(texts=train_eng[:subset_size],tokenizer=tokenizer)

In [5]:
encoded_tel = tokenization_in_batch(texts=train_tel[:subset_size],tokenizer=tokenizer)

In [9]:
encoded_eng = {k: v.to(device) for k, v in encoded_eng.items()}
encoded_tel = {k: v.to(device) for k, v in encoded_tel.items()}
print(len(encoded_eng) == len(encoded_tel))
print(encoded_eng["input_ids"],encoded_eng["attention_mask"])
print(encoded_tel["input_ids"],encoded_tel["attention_mask"])

True
tensor([[     0,  24116,     13,  ...,      1,      1,      1],
        [     0,  11249,     54,  ...,      1,      1,      1],
        [     0,   5596,   2843,  ...,      1,      1,      1],
        ...,
        [     0,  26832, 111001,  ...,      1,      1,      1],
        [     0,    581, 162049,  ...,      1,      1,      1],
        [     0,  14467,  17883,  ...,      1,      1,      1]],
       device='cuda:0') tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]], device='cuda:0')
tensor([[     0, 104562, 242255,  ...,      1,      1,      1],
        [     0,      6, 153306,  ...,      1,      1,      1],
        [     0, 121803,   4856,  ...,      1,      1,      1],
        ...,
        [     0, 143497,   9327,  ...,      1,      1,      1],
        [     0, 113164,   2195,  ...,      1,      1,      1],
     

## embedding

In [10]:
multi_model = AutoModel.from_pretrained("xlm-roberta-base").to(device)
batch_size = 512
def embed_text(texts_eng, multi_model, batch_size):
    embedded = []
    with torch.no_grad():
        for i in range(0,texts_eng["input_ids"].size(0),batch_size):
            batch = {k:v[i:i+batch_size] for k,v in texts_eng.items()}
            outputs = multi_model(**batch)
            embedded.append(outputs.last_hidden_state)
    return torch.cat(embedded,dim = 0)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
eng_embedding =  embed_text(encoded_eng, multi_model, batch_size)
tel_embedding = embed_text(encoded_tel,multi_model,batch_size)

In [12]:
print(eng_embedding.shape == tel_embedding.shape)
print(eng_embedding.shape)
print(tel_embedding.shape)
print(eng_embedding)
print(tel_embedding)
embedded = (eng_embedding, tel_embedding)
torch.save(embedded, "embeddings.pt")

True
torch.Size([5000, 128, 768])
torch.Size([5000, 128, 768])
tensor([[[ 1.5089e-01,  1.6035e-01,  6.2302e-02,  ..., -1.0316e-01,
           7.7910e-02, -2.0323e-02],
         [ 3.1381e-02,  4.9260e-02,  2.2708e-02,  ...,  2.9957e-01,
           3.1114e-03, -1.3808e-01],
         [ 3.1157e-02,  5.2559e-02, -1.3319e-02,  ...,  1.3891e-01,
           2.9262e-02,  5.0349e-02],
         ...,
         [ 1.7827e-02,  5.4567e-02,  1.2079e-02,  ..., -6.4096e-02,
          -1.7751e-02, -2.1020e-02],
         [ 1.7827e-02,  5.4567e-02,  1.2079e-02,  ..., -6.4096e-02,
          -1.7751e-02, -2.1020e-02],
         [ 1.7827e-02,  5.4567e-02,  1.2079e-02,  ..., -6.4096e-02,
          -1.7751e-02, -2.1020e-02]],

        [[ 1.0860e-01,  1.2716e-01,  8.4481e-02,  ..., -1.8335e-01,
           1.1010e-01,  2.0355e-03],
         [-5.3211e-02,  3.0837e-02, -1.3831e-02,  ...,  1.1601e-01,
          -4.2192e-02,  7.0549e-02],
         [-2.9637e-02, -6.3854e-02,  3.8069e-02,  ...,  7.5680e-02,
           3.

In [13]:
embedded_path = Path("/kaggle/working/embeddings.pt")
print(embedded_path)
with open(embedded_path, "rb") as f:
    eng_embedding, tel_embedding = torch.load(f)
print(eng_embedding.shape, tel_embedding.shape)

/kaggle/working/embeddings.pt
torch.Size([5000, 128, 768]) torch.Size([5000, 128, 768])


In [14]:
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

## Autoencoder model

In [15]:
import gc
from torch.utils.checkpoint import checkpoint

for name in ["multi_model", "model", "encoder", "decoder", "optimizer"]:
    if name  in globals():
        del globals()[name]

gc.collect()   
torch.cuda.empty_cache()     

In [16]:
gc.collect(); torch.cuda.empty_cache()
eng_embedding = eng_embedding.cpu()
tel_embedding = tel_embedding.cpu()

### encoder

In [19]:
class Encoder(nn.Module):
    def __init__(self,input_dim,hidden_dim):
        super().__init__()
        self.lstm = nn.LSTM(input_dim,hidden_dim,batch_first = True)

    def forward(self,x):
        output ,(hidden,cell) = self.lstm(x)
        return output,(hidden,cell)

encoder = Encoder(input_dim = 768,hidden_dim = 256).to(device)

## attention

In [20]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        seq_len = encoder_outputs.size(1)
        hidden = hidden.unsqueeze(1).repeat(1, seq_len, 1)
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)  
        return torch.softmax(attention, dim=1)

### decoder

In [21]:
class Decoder(nn.Module):
    def __init__(self,vocab_size,embedding_dim,hidden_dim,attention):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embedding_dim,padding_idx=1)
        self.lstm = nn.LSTM(embedding_dim + hidden_dim,hidden_dim,batch_first =True)
        self.fc = nn.Linear(hidden_dim*2,vocab_size)
        self.attention = attention

    def forward(self,x,hidden,cell,encoder_output):
        embedded = self.embedding(x)
        attention_weights = self.attention(hidden.squeeze(0),encoder_output)
        attention_weights = attention_weights.unsqueeze(1)
        context_vector = torch.bmm(attention_weights,encoder_output)
        lstm_input = torch.cat((embedded,context_vector),dim = 2)
        outputs,(hiddens,cells) = self.lstm(lstm_input,(hidden,cell))
        prediction = self.fc(torch.cat((outputs,context_vector),dim = 2))
        return prediction, hiddens,cells

In [22]:
Attention_layer = Attention(hidden_dim=256).to(device)
decoder = Decoder(vocab_size=tokenizer.vocab_size,embedding_dim=768,hidden_dim=256,attention=Attention_layer).to(device)

In [23]:
class Seq2Seq(nn.Module):
    def __init__(self,encoder,decoder,pad_token_id = 1):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.pad_token_id =pad_token_id

    def forward(self,src_embedded,tel_batch):
        encoder_outputs ,(hidden,cell)  = self.encoder(src_embedded)
        seq_length = tel_batch.size(1)
        outputs = []
        x =tel_batch[:,0:1]
        for i in range(1,seq_length):
            prediction, hidden, cell = checkpoint(self.decoder, x, hidden, cell, encoder_outputs, use_reentrant=False)
            outputs.append(prediction)
            x = tel_batch[:,i:i+1]

        outputs = torch.cat(outputs,dim = 1)
        return outputs    

In [24]:
model = Seq2Seq(encoder,decoder).to(device)
device_count = torch.cuda.device_count()
print("gpus available:",device_count)
if device_count > 1:
    model = nn.DataParallel(model,device_ids=[0,1])
    print(f"model on {device_count} gpus")

else:
    print("model on gpu")    

gpus available: 2
model on 2 gpus


## criterion and optimizer

In [29]:
criterion = nn.CrossEntropyLoss(ignore_index=1)
optimizer = optim.Adam(list(encoder.parameters())+ list(decoder.parameters()))
scaler = GradScaler("cuda")

## training

In [ ]:
def train(model,eng_embedded,tel_ids,batch, batch_size = 128):
    model.train()
    total_loss,n_batches = 0,0
    n = eng_embedded.size(0)

    for i in range(0,n,batch_size):
        src_embedded = eng_embedded[i:i+batch_size].to(device)
        tel_batch_full = tel_ids[i:i+batch_size].to(device)
        mask_batch = batch[i:i+batch_size].to(device)
        length = mask_batch.sum(dim = 1).max().item()
        tel_batch = tel_batch_full[:,:length]
        optimizer.zero_grad()
        with autocast("cuda"):
            outputs = model(src_embedded,tel_batch)
            tragets = tel_batch[:,1:]
            loss = criterion(outputs.reshape(-1,outputs.size(-1)),tragets.reshape(-1))
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        n_batches += 1
        del outputs, loss, src_embedded,tel_batch_full, tel_batch,mask_batch
        torch.cuda.empty_cache()
    return total_loss / n_batches

avg_loss = train(model, eng_embedding, encoded_tel["input_ids"], encoded_tel["attention_mask"], batch_size=8)
print(f"Average training loss: {avg_loss:.4f}")